In [1]:
# # 安装vllm
# !pip uninstall --yes "tensorflow" "matplotlib" "keras" "scikit-learn" "protobuf" 
# !pip uninstall --yes "numpy" "torch"

# !pip install --no-index --find-links={install_path} vllm

In [2]:
# !pip install flash-attn

# Step 1 依赖 & 超参数

In [3]:
# 1.1 依赖

# llm inference
from transformers import pipeline, AutoTokenizer # transformers = 5.0.0
# from transformers import BitsAndBytesConfig
# from vllm import LLM, SamplingParams

# python tool integrated reasoning
import tempfile
import time
import subprocess
import re

# Kaggle
import kaggle_evaluation.aimo_3_inference_server
import polars as pl
from typing import Optional, List

# general
import torch
import os
import numpy as np 
import pandas as pd
from collections import Counter, defaultdict

import warnings
warnings.simplefilter('ignore')

In [4]:
# #fixed seed to get similar score
# from transformers import set_seed
# set_seed(42)
# pd.set_option('display.max_colwidth', None)

In [5]:
# 1.2 环境变量

# vllm环境变量 
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"

# CUDA环境变量
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [6]:
# 1.3 模型
# DeepSeek-R1-Distill-Qwen-7B
MODEL_PATH_COT = "/kaggle/input/models/deepseek-ai/deepseek-r1/transformers/deepseek-r1-distill-qwen-14b/2"
# NuminaMath-TIR-7B
MODEL_PATH_TIR = "/kaggle/input/models/patrickchenshenyi/numinamath-tir-new/transformers/default/1/NuminaMath-7B-TIR"


In [7]:
# 1.4 本地测试
validation_set: str     # One of AI-MO/aimo-validation-amc, AI-MO/aimo-validation-aime, AI-MO/aimo-validation-math-level-4, AI-MO/aimo-validation-math-level-5
is_submission: bool = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

In [8]:
# 1.5 自定义类型
MessagesBatch = list[list[dict[str, str]]]

In [9]:
# 1.6 时间限制
GLOBAL_START_TIME = time.time()
COMPETITION_TIME_LIMIT = 4.75 * 3600 #4h 45min

# Step 2. 加载模型

In [10]:
# 2.1 加载模型（pipeline）

# 不能用 question-answering (QA): 提取式 (Extractive) 任务，从给定的“上下文 (Context)”中截取一段文字作为答案
pipeline_cot = pipeline(
    "text-generation",
    model=MODEL_PATH_COT,
    model_kwargs={
        "dtype": torch.bfloat16,
        # "attn_implementation": "flash_attention_2",
        "attn_implementation": "sdpa", # Native PyTorch memory optimization
    },
    device_map="auto",
    return_full_text=False, # 重要：只返回生成的部分，不包含原始 Prompt
)
tokenizer_cot = AutoTokenizer.from_pretrained(MODEL_PATH_COT, trust_remote_code=True)
tokenizer_cot.padding_side = "left" # Left padding is required for batched batched generation in decoder-only models
if tokenizer_cot.pad_token is None:
    tokenizer_cot.pad_token = tokenizer_cot.eos_token


pipeline_tir = pipeline(
    "text-generation",
    model=MODEL_PATH_TIR,
    model_kwargs={
        "dtype": torch.bfloat16,
        # "attn_implementation": "flash_attention_2",
        "attn_implementation": "sdpa", # Native PyTorch memory optimization
    },
    device_map="auto", 
    return_full_text=False, 
)
tokenizer_tir = AutoTokenizer.from_pretrained(MODEL_PATH_TIR, trust_remote_code=True)
tokenizer_tir.padding_side = "left" 
if tokenizer_tir.pad_token is None:
    tokenizer_tir.pad_token = tokenizer_tir.eos_token


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

In [11]:
# 2.2 生成参数
# DeepSeek-R1 推荐的生成参数
GENERATION_CONFIG_COT = {
    "do_sample": True,
    "temperature": 0.6,           # R1通常用稍低的温度
    "top_p": 0.95,
    "top_k": 50,                   # R1推荐使用top_k
    "repetition_penalty": 1.1,     # 稍微提高防止重复
    "max_new_tokens" : 4096,
    "max_length": None,
}

GENERATION_CONFIG_TIR = {
    "do_sample": True,            
    "temperature": 0.4,     
    "top_p": 0.95,
    "top_k": 10, 
    "max_new_tokens": 2048,        # Generate chunks, not the whole 20k limit at once
    "max_length": None,
}

TIR_STOP_SEQUENCES = ["```output\n"] 

In [12]:
# 2.3 prompts
SYSTEM_PROMPT_COT_1 = (
    "You are a helpful mathematics expert. "
    "Solve the problem by reasoning step-by-step. "
    "Put your final answer within \\boxed{}."
)

SYSTEM_PROMPT_COT_2 = (
    "You are a creative mathematical problem solver. "
    "Before calculating, briefly brainstorm multiple ways to approach the problem (e.g., algebraic, geometric, combinatorial). "
    "Choose the most elegant path and solve it step-by-step. "
    "Put your final answer within \\boxed{}."
)

SYSTEM_PROMPT_COT_3 = (
    "You are a meticulous mathematician. "
    "Solve the problem by breaking it down into explicit sub-goals. "
    "Double-check your logic and arithmetic at each step to prevent errors. "
    "Put your final answer within \\boxed{}."
)

SYSTEM_PROMPT_COT_4 = (
    "You are an expert at mathematical logic and problem-solving. "
    "Analyze the constraints of the problem carefully. Try working backwards from the goal or using first principles to derive the solution. "
    "Put your final answer within \\boxed{}."
)

SYSTEM_PROMPT_COT = [
    SYSTEM_PROMPT_COT_1, 
    SYSTEM_PROMPT_COT_1, 
    SYSTEM_PROMPT_COT_2,  
    SYSTEM_PROMPT_COT_2, 
    SYSTEM_PROMPT_COT_3, 
    SYSTEM_PROMPT_COT_3, 
    SYSTEM_PROMPT_COT_4,
    SYSTEM_PROMPT_COT_4,
]

SYSTEM_PROMPT_TIR_1 = (
    "You are an expert mathematical AI. "
    "Solve the problem by writing a Python script to calculate the answer. "
    "Always wrap your code inside ```python\n...\n``` blocks. "
    "After observing the output, put your final answer within \\boxed{}."
)

SYSTEM_PROMPT_TIR_2 = (
    "You are an expert in computational mathematics. "
    "Solve the problem algebraically using Python's `sympy` library. Define your variables, set up the equations, and solve them programmatically. "
    "Always wrap your code inside ```python\n...\n``` blocks. "
    "After observing the output, put your final answer within \\boxed{}."
)

SYSTEM_PROMPT_TIR_3 = (
    "You are a highly logical algorithmic solver. "
    "If the problem involves sequences, combinatorics, or optimization, write a Python script using iterative loops, dynamic programming, or logical search to find the exact answer. "
    "Always wrap your code inside ```python\n...\n``` blocks. "
    "After observing the output, put your final answer within \\boxed{}."
)

SYSTEM_PROMPT_TIR_4 = (
    "You are an expert Python coder for math. "
    "Write a Python script to solve the problem. If possible, write your code to calculate the answer in two different ways to assert correctness. "
    "Always wrap your code inside ```python\n...\n``` blocks. "
    "After observing the output, put your final answer within \\boxed{}."
)

SYSTEM_PROMPT_TIR = [
    SYSTEM_PROMPT_TIR_1, 
    SYSTEM_PROMPT_TIR_2, 
    SYSTEM_PROMPT_TIR_3, 
    SYSTEM_PROMPT_TIR_4
]

In [13]:
# 2.4 测试 cot
problem = "What is $0\times10$?"
messages_cot = [
    {"role": "system", "content": SYSTEM_PROMPT_COT[0]},
    {"role": "user", "content": problem}
]
text_input_cot = tokenizer_cot.apply_chat_template(messages_cot, tokenize=False, add_generation_prompt=True)
output_cot = pipeline_cot(text_input_cot, 
                          **GENERATION_CONFIG_TIR,
                          stop_strings=TIR_STOP_SEQUENCES,
                          tokenizer=pipeline_cot.tokenizer
)

text_generated_cot = output_cot[0]['generated_text']

print(text_generated_cot)


Passing `generation_config` together with generation-related arguments=({'stop_strings', 'max_new_tokens', 'do_sample', 'top_p', 'temperature', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To solve the multiplication of 0 and 10, I start by recalling the fundamental property of zero in multiplication. This property states that any number multiplied by zero results in zero.

Next, I apply this property to the given numbers: 0 multiplied by 10. According to the property, the product of these two numbers will be zero.

Therefore, the final answer is 0.
</think>

To solve the multiplication \(0 \times 10\), let's break it down step by step:

1. **Understand the Multiplication Property of Zero:**
   
   One of the fundamental properties of multiplication is that any number multiplied by zero equals zero. This can be expressed as:
   \[
   0 \times a = 0 \quad \text{for any real number } a
   \]

2. **Apply the Property to the Given Numbers:**
   
   In this case, we have:
   \[
   0 \times 10
   \]
   According to the property, multiplying any number by zero results in zero.

3. **Calculate the Result:**
   
   Applying the property:
   \[
   0 \times 10 = 0
   \]

**Final An

In [14]:
# 2.4 测试 tir
problem = """what is the result of 1+2*3?"""
messages_tir = [
    {"role": "system", "content": SYSTEM_PROMPT_TIR[0]},
    {"role": "user", "content": problem}
]
text_input_tir = tokenizer_tir.apply_chat_template(messages_tir, tokenize=False, add_generation_prompt=True)
output_tir = pipeline_tir(text_input_tir, 
                          **GENERATION_CONFIG_TIR,
                          stop_strings=TIR_STOP_SEQUENCES,
                          tokenizer=pipeline_tir.tokenizer
)

text_generated_tir = output_tir[0]['generated_text']

print(text_generated_tir)


Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1+2*3 is an arithmetic expression that can be solved following the order of operations, often remembered by the acronym PEMDAS (Parentheses, Exponents, Multiplication and Division, Addition and Subtraction). 

Let's solve it step-by-step:

1. First, perform the multiplication: \(2 \times 3\)
2. Then, perform the addition: \(1 + \text{result of step 1}\)

Let's use Python to ensure the result is accurate.
```python
# Step-by-step calculation
# First, perform the multiplication
multiplication_result = 2 * 3

# Then, perform the addition
final_result = 1 + multiplication_result
print(final_result)
```
```output



# Step 3 Tool Integrated Reasoning

In [15]:
# 3.1 提取python块
def extract_python_code(text: str) -> list[str]:
    """
    要求：
    - messages[-1]["generated_text"]需要以
    """
    pattern = r'```python\s*(.*?)\s*```'
    # Find all python code segments in the text
    matches = re.findall(pattern, text, re.DOTALL)
    return matches


In [16]:
# 3.2 处理成可执行的python代码
def process_python_code(query):
    query = "import math\nimport numpy as np\nimport sympy as sp\n" + query
    current_rows = query.strip().split("\n")
    ans = "\n".join(current_rows)
    return ans

In [17]:
# 3.3 python代码的执行单元
class PythonREPL:
    def __init__(self, timeout=4):
        self.timeout = timeout

    def __call__(self, query):
        with tempfile.TemporaryDirectory() as temp_dir:
            temp_file_path = os.path.join(temp_dir, "tmp.py")
            with open(temp_file_path, "w", encoding="utf-8") as f:
                f.write(query)
            
            try:
                result = subprocess.run(
                    ["python3", temp_file_path],
                    capture_output=True,
                    check=False,
                    text=True,
                    timeout=self.timeout,
                )
            except subprocess.TimeoutExpired:
                error_msg = (
                    f"Execution timed out after {self.timeout} seconds. "
                    "Your code is trapped in an infinite loop or is too computationally expensive. "
                    "DO NOT try to brute-force large numbers. Please optimize your algorithm, "
                    "look for a pattern, or find a closed-form mathematical solution."
                )
                return False, error_msg

            stdout = result.stdout.strip()
            stderr = result.stderr.strip()

            if result.returncode == 0:
                return True, stdout
            else:
                # Process the error message to remove the temporary file path
                # This makes the error message cleaner and more user-friendly
                error_lines = stderr.split("\n")
                cleaned_errors = []
                for line in error_lines:
                    if temp_file_path in line:
                        # Remove the path from the error line
                        line = line.replace(temp_file_path, "<temporary_file>")
                    cleaned_errors.append(line)
                cleaned_error_msg = "\n".join(cleaned_errors)
                # Include stdout in the error case
                combined_output = f"{stdout}\n{cleaned_error_msg}" if stdout else cleaned_error_msg
                return False, combined_output

In [18]:
# 3.4 python块主执行流程

def has_boxed_answer(text:str) -> bool:
    pattern = r'oxed{(.*?)}'
    matches = re.findall(pattern, text)
    if len(matches) > 0:
        return True
    else:
        return False


def batch_execute(msgs_batch: MessagesBatch):
    """
    - 提取python块
    - 处理成可执行代码
    - 执行python代码
    - 如果有结果，原地修改messages，结尾格式：f"```output\n {code_result}\n```"
    """
    for (idx, messages) in enumerate(msgs_batch):
        last_content = messages[-1]['content']
        
        # case 1 : already generated an answer (\\boxed{...}) 
        # -> skip this message
        if has_boxed_answer(last_content):
            print(f"[Msg {idx}] Boxed answer found")
            continue
        
        # case 2 : no python block or answer
        # -> tell the model to generate python block or give an answer
        python_code_list = extract_python_code(last_content)
        if(len(python_code_list) < 1):
            error_feedback = "\n```output\nError: No python code block found. Please write python code to solve the problem.\n```"
            messages[-1]['content'] += error_feedback
            print(f"[Msg {idx}] No python code block found")
            continue

        # case 3 : generated a python block
        # -> only execute the last python block
        # -> tell the model the result 
        
        python_code = python_code_list[-1]
        python_code = process_python_code(python_code)
        print(f'[Msg {idx}] Processed python code: \n{python_code}\n')
        try:
            success, output = PythonREPL()(python_code)
            
            output = output if success else f"Error: {output}"
            
            # wrap output in the format: f"```output\n {output}\n```"
            messages[-1]['content'] += f"\n{output}\n"
            print(f'[Msg {idx}] python code output : {output}')
            
        except Exception as e:
            error_msg = str(e)
            messages[-1]['content'] += f"\nError: {error_msg}\n"
            print(f'[Msg {idx}] Execution exception: {error_msg}')

    return msgs_batch

# Step 4 单个解题流程

In [19]:
# 4.1 
def build_messages_batch(problem:str, prompts:list[str]):
    """
    - convert to [[{"role" : "..."}, {"content" : "..."}], ...]
    """
    msgs_batch: MessagesBatch = [
        [
            {"role": "system", "content": prompt},
            {"role": "user", "content": problem}
        ] for prompt in prompts
    ]
    return msgs_batch

In [20]:
# 4.2 
def batch_generate(msgs_batch: MessagesBatch, tokenizer, pipeline, generation_config: dict, stop_sequences: list[str] = None) -> MessagesBatch:
    """
    - apply chat template
    - pipeline(batch)
    - append result
    - (for TIR) cleanly stripping stop strings if present
    """
    # 1. apply chat template
    text_inputs_batch = [
        tokenizer.apply_chat_template(
            msg,
            tokenize=False,
            add_generation_prompt=True,
        )
        for msg in msgs_batch
    ]

    # 2. Set up pipeline arguments
    kwargs = {**generation_config}
    if stop_sequences: # TIR case
        kwargs["stop_strings"] = stop_sequences
        kwargs["tokenizer"] = tokenizer # stop_strings require the tokenizer to be passed

    # 3. pipeline generate
    outputs_batch = pipeline(
        text_inputs_batch,
        batch_size=len(text_inputs_batch),
        **kwargs
    )
    
    # 4. modify messages in place
    for messages, single_prompt_output in zip(msgs_batch, outputs_batch):
        generated_text = single_prompt_output[0]["generated_text"]
        
        # Clean up the trailing stop string so we don't pollute the context
        if stop_sequences:
            for stop_seq in stop_sequences:
                if generated_text.endswith(stop_seq):
                    # Strip the stop string off the end
                    generated_text = generated_text[:-len(stop_seq)]
                    break
                    
        messages.append({'role': 'assistant', 'content': generated_text})

    # print generated texts
    for (idx, messages) in enumerate(msgs_batch):
        last_content = messages[-1]['content']
        print(f"[Msg {idx}] : \n {last_content}\n")        
    
    return msgs_batch

In [21]:
# 4.3 
def extract_boxed_answers(text: str) -> list[int]:
    """
    提取 \boxed{ans} 格式的答案
    """
    pattern = r'oxed{(.*?)}'
    matches = re.findall(pattern, text)
    if not matches:
        return []
        
    ans = []
    for content in matches:
        if content.isdigit():
            # Answer contains only digits already -> record
            num = content
        else:
            # Otherwise, there are other symbols
            # --> Use `re` to find all matches and
            # extract the last one
            nums = re.findall(r'\d+', content)
            if not nums:
                # Skip if no numbers were found
                continue 
            num = nums[-1]
        ans.append(int(num))
    return ans

In [22]:
# 4.4
def batch_extract_answer(msgs_batch: MessagesBatch) -> tuple[MessagesBatch, list[int]]:
    """
    从每个messages的最后一次回答中尝试提取最终答案
    
    有答案：
    - 记录最后一个答案的数值

    没有答案：
    - 将该messages添加到下一轮需要继续生成的messages_batch中
    
    输出：
    - 还未生成答案的messages
    - 已经生成的答案
    """
    
    extracted_answers: list[int] = []
    msgs_to_keep: MessagesBatch = []
    for (idx, messages) in enumerate(msgs_batch):
        answers = extract_boxed_answers(messages[-1]['content'])
        if answers:
            ans = answers[-1]
            extracted_answers.append(ans)
            print(f"[Msg {idx}] Answer : {ans}")
        else:
            msgs_to_keep.append(messages)
            print(f"[Msg {idx}] Answer Not Found")
    return msgs_to_keep, extracted_answers

In [23]:
# 4.5 truncate messages history
def keep_latest_round(msgs_batch: MessagesBatch) -> MessagesBatch:
    """
    Truncates the message history to prevent context window collapse.
    Retains only:
      - messages[0]: The System Prompt
      - messages[1]: The Original Problem (User)
      - messages[-1]: The most recent Assistant generation (with Python output)
    """
    for messages in msgs_batch:
        # If the length is greater than 3, we have accumulated historical rounds
        if len(messages) > 3:
            # Rebuild the list in-place, dropping the middle history
            messages[:] = [messages[0], messages[1], messages[-1]]
            
    return msgs_batch

In [24]:
# 4.6 add user prompt
def add_user_prompt(msgs_batch: MessagesBatch) -> MessagesBatch:
    for messages in msgs_batch:
        # wrap output in the format: f"```output\n {output}\n```"
        # appending as a NEW user message to maintain alternating roles
        feedback = "Please analyze the results above and continue."
        messages.append({'role': 'user', 'content': feedback})

    return msgs_batch

In [25]:
# 4.7 majority voting
def select_answer(answers: list):
    valid_answers = []
    for answer in answers:
        try:
            # Disregard answers that are not integers by
            # comparing their float and int values.
            if int(answer) != float(answer):
                continue

            # Check if the int answer is between 0 and 99999, as 
            # per AIMO3 competition rules.
            if 0 <= int(answer) <= 99999:
                valid_answers.append(int(answer))
        except:
            pass # Skip conversion errors (i.e., converting text to an int for float)
    
    # As a last resort, just guess a number instead :)
    if not valid_answers:
        print("Guessing random number :)")
        return -1
    # Extract the most frequent answer from the Counter object.
    # NOTE: Counter.most_common breaks ties in order of the first element occurring, so be wary of that!
    # (i.e., you could make this deterministic by sorting the valid_answer list first)
    answer, _ = Counter(valid_answers).most_common(1)[0]
    # Answer was already checked to be in the correct range.
    return answer

In [26]:
# 4.8
def solve_one_problem(problem : str, max_rounds : int = 4, max_time : int = 900):
    start_time = time.time()

    # 1. build messages batch
    msgs_batch_cot = build_messages_batch(problem, SYSTEM_PROMPT_COT)
    msgs_batch_tir = build_messages_batch(problem, SYSTEM_PROMPT_TIR)

    # 2. loop : generate -> execute -> extract answer -> modify messages
    ans_all = []
    for round_idx in range(max_rounds):
        if time.time() - start_time > max_time:
            print("================== Time Limit Exceeded =====================")
            break
        
        print(f"--------------------Round {round_idx}--------------------")

        # 3. generate text
        print(f"----------------Generating CoT ------------------")
        if msgs_batch_cot:
            msgs_batch_cot = batch_generate(
                msgs_batch_cot, 
                tokenizer_cot, 
                pipeline_cot, 
                GENERATION_CONFIG_COT
            ) 
        print(f"----------------Generating TIR ------------------")
        if msgs_batch_tir:
            msgs_batch_tir = batch_generate(
                msgs_batch_tir, 
                tokenizer_tir, 
                pipeline_tir, 
                GENERATION_CONFIG_TIR, 
                stop_sequences=TIR_STOP_SEQUENCES
            )
    
        # 4. execute code
        print(f"----------------Executing Codes -----------------")
        if msgs_batch_tir:
            msgs_batch_tir = batch_execute(msgs_batch_tir)

        # 5. extract answers; update messages batch
        print(f"----------------Extracting Ans ------------------")
        if msgs_batch_cot:
            msgs_batch_cot, ans_cot = batch_extract_answer(msgs_batch_cot)
            ans_all.extend(ans_cot)
            # Truncate history for the next round
            msgs_batch_cot = keep_latest_round(msgs_batch_cot)
            msgs_batch_cot = add_user_prompt(msgs_batch_cot)
            
        if msgs_batch_tir:
            msgs_batch_tir, ans_tir = batch_extract_answer(msgs_batch_tir)
            ans_all.extend(ans_tir)
            # Truncate history for the next round
            msgs_batch_tir = keep_latest_round(msgs_batch_tir)
            msgs_batch_tir = add_user_prompt(msgs_batch_tir)

        # 6. break if all done
        if (not msgs_batch_cot) and (not msgs_batch_tir):
            break
    
    # 7. Apply majority voting over ALL extracted answers (from python AND boxed)
    ans_final = select_answer(ans_all)
    print(f"!!!!!!!!!!!!!!!!Final Answer!!!!!!!!!!!!!!!!!!")
    print("all answers:", ans_all)
    print("final answer:",ans_final)
    return ans_final
    
    

# Step 5 Setup for Kaggle


In [27]:
# 5.1 kaggle inference server 调用的接口
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    id_ = id_.item(0)

    # running out of time
    time_elapsed = time.time() - GLOBAL_START_TIME
    if time_elapsed > COMPETITION_TIME_LIMIT:
        print(f"!!! GLOBAL TIMEOUT IMMINENT !!! Time elapsed: {time_elapsed/3600:.2f} hours. Returning fallback.")
        return pl.DataFrame({'id': id_, 'answer': 0})

    # solve single problem
    print("================== Solving Single Problem ==================")
    print(id_)
    question_str = question.item(0)
    print(question_str)
    answer = solve_one_problem(question_str)
    print("============================================================\n\n")
    return pl.DataFrame({'id': id_, 'answer': answer})

In [ ]:
# 5.2 kaggle inference server
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(
    predict
)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv',)
    )

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'top_p', 'repetition_penalty', 'temperature', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


================== Solving Single Problem ==================
9c1c5f
Let $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ be a function such that for all positive integers $m$ and $n$, 
\begin{equation*}
    f(m) + f(n) = f(m + n + mn).
\end{equation*}
Across all functions $f$ such that $f(n) \leq 1000$ for all $n \leq 1000$, how many different values can $f(2024)$ take?
--------------------Round 0--------------------
----------------Generating CoT ------------------


Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Msg 0] : 
 <think>
Okay, so I have this problem where there's a function \( f: \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1} \), which means it takes positive integers and maps them to positive integers. The condition given is that for any two positive integers \( m \) and \( n \), the equation \( f(m) + f(n) = f(m + n + mn) \) holds. Additionally, we're told that \( f(n) \leq 1000 \) for all \( n \leq 1000 \). The question is asking how many different values \( f(2024) \) can take across all such functions.

Alright, let me try to break this down step by step.

First off, functional equations can sometimes be tricky, but often they require finding patterns or specific forms of functions that satisfy the given conditions. In this case, the functional equation is:

\[ f(m) + f(n) = f(m + n + mn) \]

Hmm, that looks interesting. Let me see if I can manipulate this equation somehow. Maybe substituting specific values for \( m \) and \( n \) will help me figure out what kind of function \( 

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[Msg 2] python code output : 
----------------Extracting Ans ------------------
[Msg 0] Answer : 1
[Msg 0] Answer Not Found
[Msg 1] Answer Not Found
[Msg 2] Answer Not Found
!!!!!!!!!!!!!!!!Final Answer!!!!!!!!!!!!!!!!!!
all answers: [500, 111, 125, 167, 125, 500, 0, 12, 1]
final answer: 500


================== Solving Single Problem ==================
641659
Let $ABC$ be a triangle with $AB \neq AC$, circumcircle $\Omega$, and incircle $\omega$. Let the contact points of $\omega$ with $BC$, $CA$, and $AB$ be $D$, $E$, and $F$, respectively. Let the circumcircle of $AFE$ meet $\Omega$ at $K$ and let the reflection of $K$ in $EF$ be $K'$. Let $N$ denote the foot of the perpendicular from $D$ to $EF$. The circle tangent to line $BN$ and passing through $B$ and $K$ intersects $BC$ again at $T \neq B$. 
    
Let sequence $(F_n)_{n \geq 0}$ be defined by $F_0 = 0$, $F_1 = 1$ and for $n \geq 2$, $F_n = F_{n-1} + F_{n-2}$. Call $ABC$ $n$\emph{-tastic} if $BD = F_n$, $CD = F_{n+1}$, and $KNK'

Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Msg 0] : 
 <think>

Alright, so I'm trying to solve this geometry problem about an n-tastic triangle ABC. It's kind of complex, but I'll try breaking it down step by step.

First, let me understand the definitions given:

1. **Triangle ABC**: Not isoceles (since AB ≠ AC). Its circumcircle is called Ω, and its incircle is ω.
2. **Contact Points D, E, F**: These are where the incircle ω touches sides BC, CA, and AB respectively.
3. **Circumcircle of AFE**: This meets the circumcircle Ω again at point K.
4. **Reflection of K over EF**: That's point K'.
5. **Foot of Perpendicular from D to EF**: That's N.
6. **Circle Tangent to BN Passing Through B and K**: This intersects BC again at T ≠ B.

Then, there's a Fibonacci-like sequence Fₙ defined as:
- F₀ = 0,
- F₁ = 1,
- For n ≥ 2, Fₙ = Fₙ₋₁ + Fₙ₋₂.

An n-tastic triangle has BD = Fₙ, CD = Fₙ₊₁, and quadrilateral KNK'B is cyclic. We need to find aₙ, which is the maximum possible value of (CT · NB)/(BT · NE), across all n-tastic triangles. The

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Msg 3] python code output : 0
----------------Extracting Ans ------------------
[Msg 0] Answer Not Found
[Msg 1] Answer Not Found
[Msg 2] Answer Not Found
[Msg 3] Answer Not Found
[Msg 4] Answer Not Found
[Msg 5] Answer Not Found
[Msg 6] Answer Not Found
[Msg 7] Answer Not Found
[Msg 0] Answer Not Found
[Msg 1] Answer Not Found
[Msg 2] Answer Not Found
[Msg 3] Answer Not Found
--------------------Round 1--------------------
----------------Generating CoT ------------------


In [ ]:
print("Done")